# 06 — Data Normalization: Unified Project Model

**Objective**: assemble the full `Project` model (Section 3) for every entry in `config/project_mapping.yaml` by joining the Jira layer (notebooks 02/03) and the financial layer (notebook 04), and cross-check every figure against those earlier notebooks to prove the unifier isn't quietly recomputing anything differently.

**Dependencies**: `src/services/project_unifier.py`.

**Scope boundary**: this notebook assembles FACTS only. `risk_status` stays `UNKNOWN` for every project here — turning these facts into a RAG judgment is Phase 6's risk engine, not this layer's job.

In [1]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

from datetime import date
import pandas as pd

from src.connectors.jira_client import build_default_jira_client
from src.connectors.financial_client import CSVFinancialDataSource
from src.services import project_unifier

jira_client = build_default_jira_client()
fin_source = CSVFinancialDataSource()
mapping = project_unifier.load_project_mapping()
AS_OF = date(2026, 9, 15)

portfolio = project_unifier.build_unified_portfolio(mapping, jira_client, fin_source, AS_OF)
print(f"{len(portfolio)} unified Project records built")

7 unified Project records built


## The unified portfolio

In [2]:
rows = []
for p in portfolio:
    rows.append({
        "project_id": p.project_id,
        "key": p.project_key,
        "delivery_progress_pct": round(p.delivery_progress_pct, 1) if p.delivery_progress_pct is not None else None,
        "delivery_status": p.delivery_status,
        "approved_budget": round(p.approved_budget, 2) if p.approved_budget is not None else None,
        "budget_consumption_pct": round(p.budget_consumption_pct, 1) if p.budget_consumption_pct is not None else None,
        "financial_status": p.financial_status,
        "risk_status": p.risk_status.value,
    })
pd.DataFrame(rows)

,project_id,key,delivery_progress_pct,delivery_status,approved_budget,budget_consumption_pct,financial_status,risk_status
0,PROJECT-10001,PHX,37.9,NaN,60533.77,111.9,NaN,UNKNOWN
1,PROJECT-10002,ORCA,30.4,NaN,77273.10,81.0,NaN,UNKNOWN
2,PROJECT-10003,NOVA,36.1,NaN,62946.26,113.7,NaN,UNKNOWN
3,PROJECT-10004,TITAN,49.1,NaN,71911.33,102.4,NaN,UNKNOWN
4,PROJECT-10005,LYNX,45.3,NaN,37219.86,106.2,NaN,UNKNOWN
5,PROJECT-10006,QSR,64.3,NaN,NaN,NaN,UNKNOWN,UNKNOWN
6,PROJECT-10007,FIN-10007,NaN,UNKNOWN,50500.00,64.6,NaN,UNKNOWN


Every `risk_status` reads `UNKNOWN` — deliberately. This model has facts, not a verdict yet.

## Cross-check: unified figures match the source-specific notebooks exactly

This is the actual point of the exercise — proving the join doesn't introduce a second, silently-different calculation path.

In [3]:
from src.services import sprint_metrics, reconciliation

phx = next(p for p in portfolio if p.project_id == "PROJECT-10001")

# Delivery: recompute directly from the Jira layer (notebook 03's method)
direct_issues = jira_client.get_project_issues("PHX", as_of=AS_OF).records
direct_progress = sprint_metrics.overall_delivery_progress_pct(direct_issues)
print(f"Unified delivery_progress_pct: {phx.delivery_progress_pct}")
print(f"Direct recomputation:          {direct_progress}")
assert phx.delivery_progress_pct == direct_progress

# Financial: recompute directly from the financial layer (notebook 04's method)
latest_period = fin_source.get_latest_reporting_period("10001")
direct_eval = reconciliation.evaluate_financial_record(fin_source, "10001", latest_period, as_of=AS_OF)
print(f"\nUnified approved_budget:   {phx.approved_budget}")
print(f"Direct recomputation:      {direct_eval.record.approved_budget}")
assert phx.approved_budget == direct_eval.record.approved_budget
assert phx.remaining_budget == direct_eval.record.remaining_budget
print("\nBoth match exactly.")

Unified delivery_progress_pct: 37.903225806451616
Direct recomputation:          37.903225806451616

Unified approved_budget:   60533.77
Direct recomputation:      60533.77

Both match exactly.


## The two mapping-gap projects, in full

One field is `None` and the matching `*_status` says `UNKNOWN` — never a fabricated 0 or an empty string standing in for missing data.

In [4]:
import json

qsr = next(p for p in portfolio if p.project_id == "PROJECT-10006")
helios = next(p for p in portfolio if p.project_id == "PROJECT-10007")

print("QSR (Jira-only):")
print(json.dumps({
    "delivery_progress_pct": qsr.delivery_progress_pct,
    "delivery_status": qsr.delivery_status,
    "approved_budget": qsr.approved_budget,
    "financial_status": qsr.financial_status,
}, indent=2))

print("\nHelios (Finance-only):")
print(json.dumps({
    "delivery_progress_pct": helios.delivery_progress_pct,
    "delivery_status": helios.delivery_status,
    "approved_budget": helios.approved_budget,
    "financial_status": helios.financial_status,
}, indent=2))

QSR (Jira-only):
{
  "delivery_progress_pct": 64.28571428571429,
  "delivery_status": null,
  "approved_budget": null,
  "financial_status": "UNKNOWN"
}

Helios (Finance-only):
{
  "delivery_progress_pct": null,
  "delivery_status": "UNKNOWN",
  "approved_budget": 50500.0,
  "financial_status": null
}


## Provenance: every fact traces back to a source record and a timestamp

In [5]:
for p in portfolio:
    print(f"{p.project_id:16s} ", end="")
    print(", ".join(f"{pr.source_system.value}:{pr.source_record_id}" for pr in p.provenance) or "(no sources — fully unmapped)")

PROJECT-10001    Jira:PHX, Finance:10001/2026-08
PROJECT-10002    Jira:ORCA, Finance:10002/2026-08
PROJECT-10003    Jira:NOVA, Finance:10003/2026-08
PROJECT-10004    Jira:TITAN, Finance:10004/2026-08
PROJECT-10005    Jira:LYNX, Finance:10005/2026-08
PROJECT-10006    Jira:QSR
PROJECT-10007    Finance:10007/2026-08


## Validation checks

- [x] All 7 mapped projects produce a `Project` object
- [x] Delivery and financial figures for a fully-mapped project match direct recomputation from the source-specific layers exactly (not approximately — `==`, not `pytest.approx`)
- [x] `QSR` and `Helios` each have exactly one side `None`/`UNKNOWN`, never both, never neither
- [x] `risk_status` is `UNKNOWN` for every project — this layer does not judge, only assembles
- [x] `provenance` lists exactly the sources actually consulted for each project (2 for fully-mapped, 1 for each gap case)

## Testing

`tests/test_project_unifier.py` (22 tests) — `pytest tests/test_project_unifier.py -v`.

## Next step

Phase 6: `src/services/risk_engine.py` — turn these facts (plus the financial risk detectors already built in `financial_metrics.py`) into `delivery_risk` / `financial_risk` / `combined_risk` per `config/risk_rules.yaml`'s matrix, finally filling in `risk_status` for real.